In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


SOURCE_DIR = Path("../data/csv")

OUTPUT_PATH = Path("../data/raw/btc_network_daily.csv")

START_DATE = "2012-01-01"
END_DATE = "2026-01-01"


def load_blockchain_csv(filename: str, value_column: str) -> pd.DataFrame:
    """
    Загружает CSV формата:
    2009-01-17 00:00:00,109.0
    """
    path = SOURCE_DIR / filename

    if not path.exists():
        raise FileNotFoundError(f"Файл не найден: {path}")

    df = pd.read_csv(
        path,
        header=None,
        names=["date", value_column]
    )

    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.floor("D")
    df[value_column] = pd.to_numeric(df[value_column], errors="coerce")

    df = df.dropna(subset=["date", value_column])
    df = df.groupby("date", as_index=False)[value_column].mean()
    df = df.sort_values("date").reset_index(drop=True)

    return df


# 1. Количество уникальных адресов
unique_addresses = load_blockchain_csv(
    "n-unique-addresses.csv",
    "unique_addresses"
)

# 2. Оценочный объем транзакций BTC
transfer_volume = load_blockchain_csv(
    "estimated-transaction-volume.csv",
    "transfer_volume_btc"
)

# 3. Суммарные комиссии за день в USD
transaction_fees_usd = load_blockchain_csv(
    "transaction-fees-usd.csv",
    "transaction_fees_usd"
)

# 4. Количество транзакций за день
n_transactions = load_blockchain_csv(
    "n-transactions.csv",
    "n_transactions"
)


network = (
    unique_addresses
    .merge(transfer_volume, on="date", how="inner")
    .merge(transaction_fees_usd, on="date", how="inner")
    .merge(n_transactions, on="date", how="inner")
)


network["avg_fee_usd"] = np.where(
    network["n_transactions"] > 0,
    network["transaction_fees_usd"] / network["n_transactions"],
    0.0
)


network = network[
    (network["date"] >= pd.to_datetime(START_DATE)) &
    (network["date"] <= pd.to_datetime(END_DATE))
].copy()


network = network[
    [
        "date",
        "unique_addresses",
        "transfer_volume_btc",
        "avg_fee_usd"
    ]
]


if network.empty:
    raise ValueError("После фильтрации по датам итоговый набор сетевых признаков пустой.")

if network.isna().any().any():
    raise ValueError(f"В итоговом наборе есть пропуски:\n{network.isna().sum()}")

if not np.isfinite(network[["unique_addresses", "transfer_volume_btc", "avg_fee_usd"]].to_numpy()).all():
    raise ValueError("В итоговом наборе есть бесконечные значения.")

if (network[["unique_addresses", "transfer_volume_btc", "avg_fee_usd"]] < 0).any().any():
    raise ValueError("В итоговом наборе есть отрицательные значения.")


network["date"] = network["date"].dt.strftime("%Y-%m-%d")


OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

network.to_csv(OUTPUT_PATH, index=False)

print(f"Файл создан: {OUTPUT_PATH}")
print(f"Количество строк: {len(network)}")
print()
print(network.head())
print()
print(network.tail())

Файл создан: ..\data\raw\btc_network_daily.csv
Количество строк: 5095

           date  unique_addresses  transfer_volume_btc  avg_fee_usd
473  2012-01-01            8531.0        201609.048310     0.003703
474  2012-01-02            8928.0        142482.534073     0.006652
475  2012-01-03            9528.0        110788.668431     0.005294
476  2012-01-04            9542.0        139580.305439     0.013625
477  2012-01-05           11636.0        278373.988174     0.006391

            date  unique_addresses  transfer_volume_btc  avg_fee_usd
5563  2025-12-28          435884.0         24025.474084     0.353997
5564  2025-12-29          516338.0        204341.069184     0.453153
5565  2025-12-30          539847.0        138538.200619     0.708147
5566  2025-12-31          547272.0        196214.688680     0.711765
5567  2026-01-01          441382.0         62095.399289     0.445932
